# RAG Pipeline
## Data Ingestion to VectorDB

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [16]:
from langchain_core.documents import Document
import os

def process_all_texts(text_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    text_dir = Path(text_directory)
    text_files = list(text_dir.glob("**/*.txt"))
    print(f"Found {len(text_files)} text files to process")
    for text_file in text_files:
        print(f"\nProcessing: {text_file.name}")
        try:
            with open(text_file, 'r', encoding='utf-8') as f:
                content = f.read()
            doc = Document(
                page_content=content,
                metadata={
                    'source_file': text_file.name,
                    'file_type': 'txt'
                }
            )
            all_documents.append(doc)
            print(f"  ✓ Loaded text file")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all text files in the data directory
all_text_documents = process_all_texts("../data")

Found 2 text files to process

Processing: machine_learning.txt
  ✓ Loaded text file

Processing: python_intro.txt
  ✓ Loaded text file

Total documents loaded: 2


In [24]:
all_text_documents

[Document(metadata={'source_file': 'machine_learning.txt', 'file_type': 'txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a subset of artificial intelligence (AI) that enables computers \nto learn and improve from experience without being explicitly programmed.\n\nTypes of Machine Learning:\n\n1. Supervised Learning\n   - Uses labeled training data\n   - Examples: Linear regression, Decision trees, Support Vector Machines\n   - Applications: Email spam detection, Image classification\n\n2. Unsupervised Learning\n   - Finds patterns in data without labeled examples\n   - Examples: K-means clustering, Principal Component Analysis\n   - Applications: Customer segmentation, Anomaly detection\n\n3. Reinforcement Learning\n   - Learns through interaction with environment\n   - Examples: Q-learning, Deep Q-Networks\n   - Applications: Game playing, Robotics, Autonomous vehicles\n\nKey Concepts:\n- Training data: Dataset used to teach the model\n- Features: Input 

In [18]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks=split_documents(all_text_documents)
chunks

Split 2 documents into 3 chunks

Example chunk:
Content: Machine Learning Fundamentals

Machine Learning (ML) is a subset of artificial intelligence (AI) that enables computers 
to learn and improve from experience without being explicitly programmed.

Type...
Metadata: {'source_file': 'machine_learning.txt', 'file_type': 'txt'}


[Document(metadata={'source_file': 'machine_learning.txt', 'file_type': 'txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a subset of artificial intelligence (AI) that enables computers \nto learn and improve from experience without being explicitly programmed.\n\nTypes of Machine Learning:\n\n1. Supervised Learning\n   - Uses labeled training data\n   - Examples: Linear regression, Decision trees, Support Vector Machines\n   - Applications: Email spam detection, Image classification\n\n2. Unsupervised Learning\n   - Finds patterns in data without labeled examples\n   - Examples: K-means clustering, Principal Component Analysis\n   - Applications: Customer segmentation, Anomaly detection\n\n3. Reinforcement Learning\n   - Learns through interaction with environment\n   - Examples: Q-learning, Deep Q-Networks\n   - Applications: Game playing, Robotics, Autonomous vehicles'),
 Document(metadata={'source_file': 'machine_learning.txt', 'file_type': 'txt'}, pag

### Embedding and VectoStoreDB

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
import requests
import numpy as np

class EmbeddingManager:
    """Handles document embedding generation using Ollama API"""
    
    def __init__(self, ollama_url: str = "http://localhost:11434/api/embeddings", model: str = "all-minilm"):
        """
        Initialize the embedding manager for Ollama
        
        Args:
            ollama_url: URL of the Ollama embeddings API endpoint
            model: Name of the embedding model to use
        """
        self.ollama_url = ollama_url
        self.model = model
    
    def generate_embeddings(self, texts: list) -> np.ndarray:
        """
        Generate embeddings for a list of texts using Ollama API
        
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        embeddings = []
        for text in texts:
            payload = {
                "model": self.model,
                "prompt": text
            }
            response = requests.post(self.ollama_url, json=payload)
            response.raise_for_status()
            data = response.json()
            if "embedding" not in data:
                raise ValueError(f"No embedding returned for text: {text}")
            embeddings.append(data["embedding"])
        embeddings = np.array(embeddings)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

In [25]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "all_text_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Text document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: all_text_documents
Existing documents in collection: 0


In [27]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generated embeddings with shape: (3, 384)
Adding 3 documents to vector store...
Successfully added 3 documents to vector store
Total documents in collection: 3


In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [29]:
rag_retriever

In [30]:
rag_retriever.retrieve("What are the Types of Machine Learning?")

Retrieving documents for query: 'What are the Types of Machine Learning?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_4cadaa8b_0',
  'content': 'Machine Learning Fundamentals\n\nMachine Learning (ML) is a subset of artificial intelligence (AI) that enables computers \nto learn and improve from experience without being explicitly programmed.\n\nTypes of Machine Learning:\n\n1. Supervised Learning\n   - Uses labeled training data\n   - Examples: Linear regression, Decision trees, Support Vector Machines\n   - Applications: Email spam detection, Image classification\n\n2. Unsupervised Learning\n   - Finds patterns in data without labeled examples\n   - Examples: K-means clustering, Principal Component Analysis\n   - Applications: Customer segmentation, Anomaly detection\n\n3. Reinforcement Learning\n   - Learns through interaction with environment\n   - Examples: Q-learning, Deep Q-Networks\n   - Applications: Game playing, Robotics, Autonomous vehicles',
  'metadata': {'doc_index': 0,
   'source_file': 'machine_learning.txt',
   'content_length': 799,
   'file_type': 'txt'},
  'similarity_s

In [31]:
rag_retriever.retrieve("What are the key features of Python")

Retrieving documents for query: 'What are the key features of Python'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_a031eb8b_2',
  'content': 'Introduction to Python Programming\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. \nIt was created by Guido van Rossum and first released in 1991.\n\nKey Features of Python:\n- Easy to learn and use\n- Interpreted language\n- Object-oriented programming support\n- Large standard library\n- Cross-platform compatibility\n- Dynamic typing\n- Automatic memory management\n\nPython is widely used in:\n- Web development\n- Data science and analytics\n- Artificial intelligence and machine learning\n- Scientific computing\n- Automation and scripting\n- Desktop application development\n\nPopular Python frameworks and libraries:\n- Django and Flask for web development\n- NumPy and Pandas for data analysis\n- TensorFlow and PyTorch for machine learning\n- Matplotlib and Seaborn for data visualization',
  'metadata': {'content_length': 828,
   'source_file': 'python_intro.txt',
   'doc_index': 2,
   'file

RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import json

import requests

def ollama_rag_response(
    query, 
    rag_retriever, 
    ollama_url="http://localhost:11434/api/generate", 
    model_name="llama3.2:3b", 
    top_k=5
):
    # Step 1: Retrieve relevant context from vector DB
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        return "No relevant context found to answer the question."

    # Step 2: Prepare prompt for LLM
    print("calling llm")
    prompt = f"""Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
    print("loading data")
    # Step 3: Call Ollama LLM API (streaming)
    payload = {
        "model": model_name,
        "prompt": prompt
    }
    print("payload prepared, making API call...")
    response = requests.post(ollama_url, json=payload, stream=True)
    response.raise_for_status()
    answer = ""
    for line in response.iter_lines():
        if line:
            try:
                data = json.loads(line.decode("utf-8"))
                answer += data.get("response", "")
            except Exception:
                continue
    return answer if answer else "No response from LLM."

In [52]:
question = "What are the types of Machine Learning?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'What are the types of Machine Learning?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
calling llm
loading data
payload prepared, making API call...
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data


In [53]:
print("Question:", question)
print("LLM Response:", response)

Question: What are the types of Machine Learning?
LLM Response: There are three main types of Machine Learning:

1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning


In [54]:
question = "What are the key concepts of Machine Learning?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'What are the key concepts of Machine Learning?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
calling llm
loading data
payload prepared, making API call...
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
ptinting data
pt

In [55]:
print("Question:", question)
print("LLM Response:", response)

Question: What are the key concepts of Machine Learning?
LLM Response: The key concepts of Machine Learning (ML) include:

1. **Supervised Learning**: A type of ML where the algorithm learns from labeled data to make predictions or decisions.

2. **Unsupervised Learning**: A type of ML where the algorithm finds patterns and structure in unlabeled data, such as customer segmentation and anomaly detection.

3. **Reinforcement Learning**: A type of ML where the algorithm learns through trial and error by interacting with an environment, used in applications like game playing and robotics.

4. **Classification**: The process of assigning a new observation to one or more categories based on features or patterns learned from data.

5. **Regression**: The process of predicting continuous values based on input features or patterns learned from data.

6. **Neural Networks**: A fundamental component of many ML models, inspired by the human brain's structure and function, used for tasks like imag